
# NeoLM — Text Generation Benchmark

Evaluate text generation quality using a fixed set of prompts and reproducible sampling settings.

## Evaluation

- Story continuation
- Factual text continuation
- Technical text continuation
- Dialogue continuation

## Metrics

- Coherence and relevance (manual evaluation)
- Repetition
- Generation length and EOS behavior
- Generation speed (tokens/sec)

The same prompts and sampling configuration will be used across comparable base language models.

**Note:** This benchmark evaluates generation behavior, not instruction-following ability. Qualitative results are subjective and should be interpreted alongside quantitative benchmarks.


In [ ]:
from pathlib import Path
import json
import time

import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed

MODEL_DIR = Path("../artifacts/model")
RESULTS_DIR = Path("results")

SEED = 1337
MAX_NEW_TOKENS = 200

GENERATION_CONFIGS = {
    "conservative": {
        "temperature": 0.3,
        "top_k": 50,
        "top_p": 0.9,
        "repetition_penalty": 1.15,
    },
    "balanced": {
        "temperature": 0.7,
        "top_k": 50,
        "top_p": 0.9,
        "repetition_penalty": 1.15,
    },
    "creative": {
        "temperature": 0.8,
        "top_k": 50,
        "top_p": 0.9,
        "repetition_penalty": 1.15,
    },
}

PROMPTS = {
    "story": [
        "Once upon a time, in a small village",
        "The young scientist opened the mysterious box and",
    ],
    "factual": [
        "The history of the Roman Empire",
        "The solar system consists of",
    ],
    "technical": [
        "Question: What is CPU stand for?",
        "The future of artificial intelligence",
        "Artificial intelligence is a field of computer science that",
        "In Python, a function is defined using",
    ],
    "dialogue": [
        'Alice: "What are you doing tomorrow?"\nBob:',
        'Teacher: "Today we will learn about gravity."\nStudent:',
    ],
}

print("=" * 60)
print("NEOLM — GENERATION BENCHMARK")
print("=" * 60)
print(f"Categories:        {len(PROMPTS)}")
print(f"Prompts:           {sum(map(len, PROMPTS.values()))}")
print(f"Generation configs:{len(GENERATION_CONFIGS):>3}")
print(f"Total generations: {sum(map(len, PROMPTS.values())) * len(GENERATION_CONFIGS)}")
print(f"Seed:              {SEED}")
print(f"Max new tokens:    {MAX_NEW_TOKENS}")
print("=" * 60)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(
    str(MODEL_DIR),
    trust_remote_code=True,
    local_files_only=True,
)

model = AutoModelForCausalLM.from_pretrained(
    str(MODEL_DIR),
    trust_remote_code=True,
    local_files_only=True,
).to(device).eval()

print("=" * 60)
print("NEOLM — MODEL LOADED")
print("=" * 60)
print(f"Parameters: {model.num_parameters():,}")
print(f"Device:     {device}")
print(f"Dtype:      {next(model.parameters()).dtype}")
print(f"KV Cache:   {model.generation_config.use_cache}")
print("=" * 60)

In [ ]:
results = []

prompts = [
    (category, prompt)
    for category, items in PROMPTS.items()
    for prompt in items
]

total_runs = len(prompts) * len(GENERATION_CONFIGS)

print("=" * 60)
print("NEOLM — RUNNING GENERATION BENCHMARK")
print("=" * 60)

for prompt_index, (category, prompt) in enumerate(
    tqdm(prompts, desc="Benchmark prompts")
):
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
    ).to(model.device)

    prompt_tokens = inputs["input_ids"].shape[1]

    max_tokens = min(
        MAX_NEW_TOKENS,
        model.config.max_position_embeddings - prompt_tokens,
    )

    assert max_tokens > 0, "Prompt exceeds context length."

    for config_name, config in GENERATION_CONFIGS.items():
        # Same seed per prompt across sampling configurations
        set_seed(SEED + prompt_index)

        if device == "cuda":
            torch.cuda.synchronize()

        start = time.perf_counter()

        with torch.inference_mode():
            output = model.generate(
                **inputs,
                max_new_tokens=max_tokens,
                do_sample=True,
                pad_token_id=tokenizer.pad_token_id,
                **config,
            )

        if device == "cuda":
            torch.cuda.synchronize()

        elapsed = time.perf_counter() - start

        generated_ids = output[0, prompt_tokens:]
        generated_tokens = len(generated_ids)

        generated_text = tokenizer.decode(
            generated_ids,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False,
        )

        stopped_by_eos = (
            generated_tokens > 0
            and generated_ids[-1].item() == tokenizer.eos_token_id
        )

        results.append({
            "category": category,
            "prompt": prompt,
            "config_name": config_name,
            "generation_config": config.copy(),
            "seed": SEED + prompt_index,
            "generated_text": generated_text,
            "prompt_tokens": prompt_tokens,
            "generated_tokens": generated_tokens,
            "stopped_by_eos": stopped_by_eos,
            "reached_token_limit": (
                generated_tokens == max_tokens and not stopped_by_eos
            ),
            "runtime_seconds": round(elapsed, 3),
            "tokens_per_second": round(
                generated_tokens / elapsed, 2
            ),
        })

# Save results
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

output_file = RESULTS_DIR / "neolm_generation.json"

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(
        {
            "model": "MaagDeveloper/NeoLM-Small-Base",
            "max_new_tokens": MAX_NEW_TOKENS,
            "results": results,
        },
        f,
        ensure_ascii=False,
        indent=2,
    )

print(f"\n{'=' * 60}")
print("GENERATION BENCHMARK COMPLETE")
print("=" * 60)
print(f"Generations:  {len(results)}/{total_runs}")
print(f"EOS stops:    {sum(r['stopped_by_eos'] for r in results)}")
print(f"Output:       {output_file}")
print("=" * 60)

In [ ]:
import pandas as pd

df = pd.DataFrame(results)

summary = df.groupby("config_name", sort=False).agg(
    generations=("generated_tokens", "size"),
    eos_stops=("stopped_by_eos", "sum"),
    avg_tokens=("generated_tokens", "mean"),
    avg_tokens_per_sec=("tokens_per_second", "mean"),
).round(2)

print("=" * 60)
print("NEOLM — GENERATION BENCHMARK SUMMARY")
print("=" * 60)

display(summary)

print("Review the generated texts for coherence,")
print("factual correctness, and repetition.")